# **Laboratorio 05**
* **Curso:** Sistema de Gestion de Datos
* **Alumno:** Maylhy Vasquez

## 1.  Subir archivos

## 2. Definir esquema esperado con StructType

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Region_Type", StringType(), True),
    StructField("Family_Size", IntegerType(), True),
    StructField("Parent_Education", StringType(), True),
    StructField("Family_Income_Level", StringType(), True),
    StructField("Internet_Quality", DoubleType(), True),
    StructField("Study_Space_Quality", DoubleType(), True),
    StructField("Previous_GPA", DoubleType(), True),
    StructField("Number_of_Failed_Courses", IntegerType(), True),
    StructField("Total_Credits_Earned", IntegerType(), True),
    StructField("Weekly_Study_Hours", DoubleType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Library_Visits_Per_Month", IntegerType(), True),
    StructField("Extracurricular_Hours", DoubleType(), True),
    StructField("Sleep_Hours", DoubleType(), True),
    StructField("Social_Media_Usage_Hours", DoubleType(), True),
    StructField("Stress_Level", DoubleType(), True),
    StructField("Motivation_Score", DoubleType(), True),
    StructField("Self_Efficacy_Score", DoubleType(), True),
    StructField("Midterm_Mark", DoubleType(), True),
    StructField("Final_Exam_Score", DoubleType(), True),
    StructField("riesgo", IntegerType(), True),
    StructField("nivel_riesgo", StringType(), True)
])

## 3. Leer archivo json 

In [0]:
from pyspark.sql.functions import col

df_estudiantes = (
    spark.read
    .format("json")
    .schema(schema)
    .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

In [0]:
display(df_estudiantes)

## 4. Leer archivos json y mapear inconsistencias

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Region_Type", StringType(), True),
    StructField("Family_Size", IntegerType(), True),
    StructField("Parent_Education", StringType(), True),
    StructField("Family_Income_Level", StringType(), True),
    StructField("Internet_Quality", DoubleType(), True),
    StructField("Study_Space_Quality", DoubleType(), True),
    StructField("Previous_GPA", DoubleType(), True),
    StructField("Number_of_Failed_Courses", IntegerType(), True),
    StructField("Total_Credits_Earned", IntegerType(), True),
    StructField("Weekly_Study_Hours", DoubleType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Library_Visits_Per_Month", IntegerType(), True),
    StructField("Extracurricular_Hours", DoubleType(), True),
    StructField("Sleep_Hours", DoubleType(), True),
    StructField("Social_Media_Usage_Hours", DoubleType(), True),
    StructField("Stress_Level", DoubleType(), True),
    StructField("Motivation_Score", DoubleType(), True),
    StructField("Self_Efficacy_Score", DoubleType(), True),
    StructField("Midterm_Mark", DoubleType(), True),
    StructField("Final_Exam_Score", DoubleType(), True)
])

In [0]:
from pyspark.sql.functions import col

df_estudiantes = (
    spark.read
    .format("json")
    .schema(schema)
    .option("multiLine", "true")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .option("badRecordsPath", "/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/bad_records/")
    .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

In [0]:
display(df_estudiantes)

## 5. try/except

In [0]:
display(df_estudiantes.select("archivo_origen").distinct())

## Encapsular la lectura en try/except

In [0]:
from pyspark.sql.functions import col

try:
    df_estudiantes = (
        spark.read
        .format("json")
        .schema(schema)
        .option("multiLine", "true")
        .option("columnNameOfCorruptRecord", "_rescued_data")
        .option("badRecordsPath", "/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/bad_records/")
        .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/")
        .withColumn("archivo_origen", col("_metadata.file_path"))
    )

    print("Lectura exitosa")

except Exception as e:
    print("Error en la lectura:", str(e))

### Separar datos válidos e inválidos

In [0]:
df_válidos = df_estudiantes.filter(col("_rescued_data").isNull())
df_invalidos = df_estudiantes.filter(col("_rescued_data").isNotNull())

In [0]:
print("No hay registros corruptos")
print(f"Total de registros válidos: {df_estudiantes.count()}")
display(df_estudiantes)

6. Crear esquema SILVER

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS proyecto_estudiantes.silver;

Guardar en Delta Tables

In [0]:
# Como no hay registros corruptos (_rescued_data no existe), todos los datos son válidos
df_estudiantes.write.format("delta").mode("overwrite").saveAsTable("proyecto_estudiantes.silver.estudiantes_validos")

print(f"✓ Tabla creada: proyecto_estudiantes.silver.estudiantes_validos ({df_estudiantes.count()} registros)")

In [0]:
# Como _rescued_data no existe, todos los registros son válidos
print("Registros válidos:", df_estudiantes.count())
print("Registros inválidos: 0")